# Day 19: Inheritance & Polymorphism
## Reuse Code with Parent and Child Classes

---

### Today's Roadmap

| Part | Topic | The Question It Answers |
|------|-------|-------------------------|
| **0** | Quick Recap | What do we already know? |
| **1** | Why Inheritance? | What problem does it solve? |
| **2** | Your First Child Class | How do I inherit from a class? |
| **3** | Adding New Methods | How does a child add its own features? |
| **4** | `super()` and `__init__` | How does a child add its own data? |
| **5** | Method Overriding | How does a child change what it inherited? |
| **6** | Polymorphism | How can one line of code work for many classes? |
| **7** | `isinstance()` & `issubclass()` | How do I check the family tree? |
| **8** | Types of Inheritance | Single, multilevel, hierarchical, multiple |
| **9** | Private Data & Inheritance | What can a child see from the parent? |
| **10** | Common Mistakes | What usually goes wrong? |
| **11** | Practice Exercises + Solutions | Can I do it myself? |
| **12** | Cheat Sheet | What do I need to remember? |

> **How to use this notebook:** Read the explanation, **guess the output**, then run the cell.

---

## Part 0: Quick Recap

| Day | What we learned |
|-----|-----------------|
| **16** | Class = blueprint, object = real thing, `__init__`, `self` |
| **17** | Instance / class / static methods, `@property`, `_protected` / `__private` |
| **18** | Magic methods like `__str__`, `__len__`, `__eq__`, `__add__` |

Today we learn how one class can **reuse** everything from another class.

---

## Part 1: Why Do We Need Inheritance?

Imagine a school app. We need a `Student` class and a `Teacher` class. Look at them carefully:

In [ ]:
# WITHOUT inheritance - lots of copy-paste!
class Student:
    def __init__(self, name, age):
        self.name = name                 # same
        self.age = age                   # same

    def introduce(self):                 # same
        return f'Hi, I am {self.name}, {self.age} years old.'

    def study(self):                     # only students
        return f'{self.name} is studying.'


class Teacher:
    def __init__(self, name, age):
        self.name = name                 # same (copied!)
        self.age = age                   # same (copied!)

    def introduce(self):                 # same (copied!)
        return f'Hi, I am {self.name}, {self.age} years old.'

    def teach(self):                     # only teachers
        return f'{self.name} is teaching.'

s = Student('Amit', 15)
t = Teacher('Mrs. Sharma', 40)
print(s.introduce())
print(t.introduce())

😰 **The problem:** `name`, `age` and `introduce()` are written **twice**.

- Found a bug in `introduce()`? Fix it in **two** places.
- Adding a `Principal` class too? Copy it a **third** time.

💡 **The solution:** Put the common parts in **one parent class** (`Person`). Then `Student` and `Teacher` **inherit** them, and only write what is **different**.

```
                 ┌────────────────────┐
                 │   Person (Parent)  │
                 │  name, age         │
                 │  introduce()       │
                 └─────────┬──────────┘
                           │  inherits
              ┌────────────┴────────────┐
              ▼                         ▼
     ┌─────────────────┐       ┌─────────────────┐
     │ Student (Child) │       │ Teacher (Child) │
     │   + study()     │       │   + teach()     │
     └─────────────────┘       └─────────────────┘
```

### 📖 Key words

| Word | Also called | Meaning |
|------|-------------|---------|
| **Parent class** | Base class, Super class | The class whose features are given |
| **Child class** | Derived class, Sub class | The class that receives the features |
| **Inheritance** | — | A child automatically gets everything the parent has |

**Real-life analogy:** You inherit your eye colour from your parents *for free*, but you also have your **own** hobbies that your parents don't have.

---

## Part 2: Your First Child Class

The syntax is simple: put the **parent's name in brackets**.

```python
class Child(Parent):
    ...
```

Let's make a child class that adds **nothing at all** (`pass`), and see what it gets for free.

In [ ]:
class Person:                            # PARENT
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def introduce(self):
        return f'Hi, I am {self.name}, {self.age} years old.'


class Student(Person):                   # CHILD - inherits from Person
    pass                                 # Nothing written here!


s = Student('Amit', 15)                  # Uses Person's __init__
print(s.name)                            # Came from Person
print(s.introduce())                     # Came from Person

🎉 `Student` is empty, but it already has `__init__`, `name`, `age` and `introduce()` — all **inherited** from `Person`.

#### 🔍 How does Python find the method?

When you call `s.introduce()`, Python searches **from the bottom up**:

```
s.introduce()
     │
     ▼
 1. Look in Student  ──►  not found
     │
     ▼
 2. Look in Person   ──►  FOUND! ✅  run it
```

Remember this search order — it explains **everything** in today's lesson.

---

## Part 3: Adding New Methods to a Child

A child can add its **own** methods. These belong **only** to the child — the parent doesn't get them.

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def introduce(self):
        return f'Hi, I am {self.name}, {self.age} years old.'


class Student(Person):
    def study(self):                     # NEW - only students have this
        return f'{self.name} is studying.'


class Teacher(Person):
    def teach(self):                     # NEW - only teachers have this
        return f'{self.name} is teaching.'


s = Student('Amit', 15)
t = Teacher('Mrs. Sharma', 40)

print(s.introduce())     # inherited
print(s.study())         # its own
print(t.introduce())     # inherited
print(t.teach())         # its own

Compare this with Part 1 — same result, **no copy-paste**! 🎉

But the inheritance only goes **one way** (parent → child):

In [ ]:
p = Person('Rahul', 30)

try:
    p.study()            # A Person is not a Student!
except AttributeError as e:
    print('Error:', e)

try:
    s.teach()            # A Student is not a Teacher!
except AttributeError as e:
    print('Error:', e)

---

## Part 4: `super()` — Adding New Data to a Child

A student also has a **roll number**. So `Student` needs its own `__init__` with an extra parameter.

### ❌ First, the wrong way

If the child writes its own `__init__`, it **replaces** the parent's one (remember the search order: Python finds the child's `__init__` first and stops). So `name` and `age` are never set!

In [ ]:
class Student(Person):
    def __init__(self, name, age, roll_no):
        self.roll_no = roll_no           # Only sets roll_no... Person's __init__ never runs!

s = Student('Amit', 15, 101)
print(s.roll_no)

try:
    print(s.introduce())                 # introduce() needs self.name...
except AttributeError as e:
    print('Error:', e)

### ✅ The right way — call the parent with `super()`

**`super()` means "my parent class".** So `super().__init__(name, age)` means:

> *"Parent, please set up `name` and `age` for me like you normally do. I'll handle the extra part."*

```python
class Student(Person):
    def __init__(self, name, age, roll_no):
        super().__init__(name, age)   # 1) Parent sets name and age
        self.roll_no = roll_no        # 2) Child sets its own extra data
```

In [ ]:
class Person:
    def __init__(self, name, age):
        print('  -> Person.__init__ running')
        self.name = name
        self.age = age

    def introduce(self):
        return f'Hi, I am {self.name}, {self.age} years old.'


class Student(Person):
    def __init__(self, name, age, roll_no):
        print('-> Student.__init__ running')
        super().__init__(name, age)      # Ask the parent to do its part
        self.roll_no = roll_no           # Then do our own part


s = Student('Amit', 15, 101)
print()
print(s.name, s.age, s.roll_no)          # All three are set!
print(s.introduce())

Look at the printed order:

```
Student('Amit', 15, 101)
        │
        ▼
Student.__init__   ──── super().__init__(name, age) ────►  Person.__init__
   sets roll_no                                              sets name, age
```

> 💡 **Rule:** If a child writes its own `__init__`, it should almost always call **`super().__init__(...)`** first.

---

## Part 5: Method Overriding

**Overriding** = the child writes a method with the **same name** as the parent's method. The child's version wins (because Python searches the child **first**).

There are two ways to do it:

| Style | What it does | How |
|-------|--------------|-----|
| **Replace** | Throw away the parent's version completely | Just write the method again |
| **Extend** | Keep the parent's version and **add** to it | Call `super().method()` inside |

### 5.1 Replace the parent's method

In [ ]:
class Animal:
    def speak(self):
        return 'Some generic animal sound'


class Dog(Animal):
    def speak(self):                     # OVERRIDE - replaces Animal's speak()
        return 'Woof!'


class Cat(Animal):
    def speak(self):                     # OVERRIDE
        return 'Meow!'


class Fish(Animal):
    pass                                 # No override -> uses Animal's version


print('Dog :', Dog().speak())
print('Cat :', Cat().speak())
print('Fish:', Fish().speak())

### 5.2 Extend the parent's method with `super()`

Sometimes the parent's method is **almost** right, and we just want to **add a little more**.

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def introduce(self):
        return f'Hi, I am {self.name}, {self.age} years old.'


class Student(Person):
    def __init__(self, name, age, roll_no):
        super().__init__(name, age)
        self.roll_no = roll_no

    def introduce(self):                                 # OVERRIDE + EXTEND
        basic = super().introduce()                      # Get the parent's sentence
        return f'{basic} My roll number is {self.roll_no}.'   # ...and add to it


p = Person('Rahul', 30)
s = Student('Amit', 15, 101)

print(p.introduce())
print(s.introduce())

---

## Part 6: Polymorphism — One Line, Many Behaviours

**Poly** = many, **morph** = forms. **Polymorphism** means:

> The **same method call** behaves **differently** depending on which object receives it.

**Real-life analogy:** Your teacher says *"Everyone, speak!"* — one instruction. The Hindi student speaks Hindi, the French student speaks French. **Same command, different results.**

This works **because of overriding**. Watch how ONE loop handles every animal:

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return '...'

class Dog(Animal):
    def speak(self):
        return 'Woof!'

class Cat(Animal):
    def speak(self):
        return 'Meow!'

class Cow(Animal):
    def speak(self):
        return 'Moo!'


animals = [Dog('Bruno'), Cat('Kitty'), Cow('Gauri'), Dog('Tommy')]

for animal in animals:
    print(f'{animal.name}: {animal.speak()}')    # ONE line works for every animal!

🔑 **Why this is powerful:** The loop doesn't use any `if` to check *"is it a dog? is it a cat?"*. Each object **knows how to speak for itself**.

### A real example: a payment system

Tomorrow the company adds a new payment type? Just write a new child class. **The `checkout` code never changes.**

In [ ]:
class Payment:
    def __init__(self, amount):
        self.amount = amount

    def process(self):
        return f'Paying Rs.{self.amount}'


class CardPayment(Payment):
    def __init__(self, amount, card_number):
        super().__init__(amount)
        self.last4 = card_number[-4:]            # Store only the last 4 digits

    def process(self):
        return f'Charged Rs.{self.amount} to card ending {self.last4}'


class UPIPayment(Payment):
    def __init__(self, amount, upi_id):
        super().__init__(amount)
        self.upi_id = upi_id

    def process(self):
        return f'Sent Rs.{self.amount} via UPI ({self.upi_id})'


class CashPayment(Payment):
    def process(self):
        return f'Please pay Rs.{self.amount} in cash at the counter'


def checkout(payment):                   # Works for ANY kind of Payment
    print('Processing...', payment.process())


checkout(CardPayment(1500, '1234-5678-9012-3456'))
checkout(UPIPayment(800, 'rahul@upi'))
checkout(CashPayment(200))

### ✏️ Quick Check — Parts 1 to 6

Guess the output before you click!

```python
class A:
    def hello(self):
        return 'Hello from A'

class B(A):
    pass

class C(A):
    def hello(self):
        return 'Hello from C'

print(B().hello())
print(C().hello())
```

<details>
<summary><b>Click to see answer</b></summary>

```
Hello from A     <- B has no hello(), so Python looks in the parent A
Hello from C     <- C overrides hello(), so C's version is found first
```

</details>

---

## Part 7: `isinstance()` and `issubclass()`

Inheritance creates an **"IS-A"** relationship:
- A `Dog` **IS AN** `Animal` ✅
- An `Animal` is **not necessarily** a `Dog` ❌

Python gives us two functions to check this:

| Function | Question it asks | Example |
|----------|------------------|---------|
| `isinstance(object, Class)` | Is this **object** of this class (or its child)? | `isinstance(bruno, Animal)` |
| `issubclass(ChildClass, ParentClass)` | Is this **class** a child of that class? | `issubclass(Dog, Animal)` |

In [ ]:
bruno = Dog('Bruno')

print('--- isinstance (checks an OBJECT) ---')
print('bruno is a Dog?   ', isinstance(bruno, Dog))       # True
print('bruno is an Animal?', isinstance(bruno, Animal))   # True  (Dog IS-A Animal)
print('bruno is a Cat?   ', isinstance(bruno, Cat))       # False

print('\n--- issubclass (checks a CLASS) ---')
print('Dog child of Animal?', issubclass(Dog, Animal))    # True
print('Animal child of Dog?', issubclass(Animal, Dog))    # False
print('Dog child of Cat?   ', issubclass(Dog, Cat))       # False

> 💡 **Fun fact:** Every class in Python secretly inherits from a built-in class called **`object`**. That's where default methods like `__str__` come from!

In [ ]:
print(issubclass(Animal, object))
print(issubclass(Dog, object))
print(issubclass(int, object))       # Even built-in types!

---

## Part 8: Types of Inheritance

```
  1) SINGLE          2) MULTILEVEL        3) HIERARCHICAL          4) MULTIPLE

    Person             Vehicle                 Animal             Flyer   Swimmer
      │                  │                  ┌────┼────┐              │       │
      ▼                  ▼                  ▼    ▼    ▼              └───┬───┘
   Student              Car               Dog   Cat  Cow                 ▼
                         │                                              Duck
                         ▼
                    ElectricCar

  one parent,       a chain:            one parent,              one child,
  one child         grandparent ->      many children            many parents
                    parent -> child
```

We already saw **single** (`Person → Student`) and **hierarchical** (`Animal → Dog, Cat, Cow`). Let's look at the other two.

### 8.1 Multilevel Inheritance — a chain

In [ ]:
class Vehicle:                               # Grandparent
    def __init__(self, brand):
        self.brand = brand

    def start(self):
        return f'{self.brand} is starting...'


class Car(Vehicle):                          # Parent
    def __init__(self, brand, model):
        super().__init__(brand)
        self.model = model

    def honk(self):
        return 'Beep beep!'


class ElectricCar(Car):                      # Child
    def __init__(self, brand, model, battery_km):
        super().__init__(brand, model)
        self.battery_km = battery_km

    def charge(self):
        return f'Charging... range {self.battery_km} km'


tesla = ElectricCar('Tesla', 'Model 3', 500)

print(tesla.start())     # from Vehicle (grandparent)
print(tesla.honk())      # from Car (parent)
print(tesla.charge())    # its own

print(isinstance(tesla, Vehicle))   # True - works through the whole chain

### 8.2 Multiple Inheritance — more than one parent

A child can inherit from **two (or more)** parents: `class Child(Parent1, Parent2):`

A **duck** can fly **and** swim, so it gets abilities from both:

In [ ]:
class Flyer:
    def fly(self):
        return 'I can fly!'


class Swimmer:
    def swim(self):
        return 'I can swim!'


class Duck(Flyer, Swimmer):              # TWO parents
    def quack(self):
        return 'Quack!'


d = Duck()
print(d.fly())       # from Flyer
print(d.swim())      # from Swimmer
print(d.quack())     # its own

#### 🔍 What if both parents have the same method?

Python checks the parents **from left to right**, in the order you wrote them. This search order is called the **MRO** (**M**ethod **R**esolution **O**rder). You can see it with `ClassName.__mro__`.

In [ ]:
class Father:
    def talent(self):
        return 'Singing (from Father)'


class Mother:
    def talent(self):
        return 'Painting (from Mother)'


class Child(Father, Mother):             # Father is written first
    pass


c = Child()
print(c.talent())                        # Father wins - he's first in the list

# The exact search order Python uses:
for cls in Child.__mro__:
    print(' ->', cls.__name__)

> ⚠️ **Tip:** Multiple inheritance is powerful but can get confusing fast. Beginners should mostly use **single** and **multilevel** inheritance.

---

## Part 9: Private Data & Inheritance

On **Day 17** we learned about `_protected` and `__private`. Now we can see what they mean for a **child** class.

| Parent attribute | Can the child use it? |
|------------------|-----------------------|
| `self.name` (public) | ✅ Yes |
| `self._salary` (protected) | ✅ Yes — this is exactly what `_` is meant for ("family only") |
| `self.__pin` (private) | ❌ No — Python renamed it to `_Parent__pin` |

In [ ]:
class Employee:
    def __init__(self, name, salary, pin):
        self.name = name                 # public
        self._salary = salary            # protected -> OK for children
        self.__pin = pin                 # private   -> ONLY for Employee


class Manager(Employee):
    def show(self):
        print('Name  :', self.name)          # ✅ works
        print('Salary:', self._salary)       # ✅ works
        try:
            print('PIN   :', self.__pin)     # ❌ Python looks for _Manager__pin
        except AttributeError as e:
            print('PIN   : Error ->', e)


m = Manager('Priya', 90000, 4321)
m.show()
print(m.__dict__)                        # See the real (mangled) name

### Why does Python do this?

Name mangling protects the parent's private data from being **accidentally overwritten** by a child that happens to use the same name:

In [ ]:
class Parent:
    def __init__(self):
        self.__secret = 'parent secret'      # stored as _Parent__secret


class Child(Parent):
    def __init__(self):
        super().__init__()
        self.__secret = 'child secret'       # stored as _Child__secret (different!)


c = Child()
print(c.__dict__)
# Both values are safe - they did NOT overwrite each other

> 💡 **Rule of thumb:** Use `_single_underscore` if child classes should be able to use it. Use `__double_underscore` only when you really want the data kept **inside this one class**.

---

## Part 10: Common Beginner Mistakes

### ❌ Mistake 1: Forgetting `super().__init__()`

We saw this in Part 4 — the parent's attributes are never created, and you get an `AttributeError` later.

### ❌ Mistake 2: Forgetting to pass the arguments to `super().__init__()`

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age


class Student(Person):
    def __init__(self, name, age, roll_no):
        super().__init__()               # Oops! Person needs name and age
        self.roll_no = roll_no

try:
    s = Student('Amit', 15, 101)
except TypeError as e:
    print('Error:', e)

# ✅ Fix:  super().__init__(name, age)

### ❌ Mistake 3: Using inheritance when it doesn't make sense

Only use inheritance when the **"IS-A"** sentence sounds correct:

| Sentence | Makes sense? | Use |
|----------|--------------|-----|
| A `Dog` **is an** `Animal` | ✅ Yes | `class Dog(Animal)` |
| A `Student` **is a** `Person` | ✅ Yes | `class Student(Person)` |
| A `Car` **is an** `Engine` | ❌ No! A car **HAS** an engine | Store it as an attribute instead |

In [ ]:
# ✅ "HAS-A" relationship -> use an attribute, NOT inheritance
class Engine:
    def start(self):
        return 'Engine started: Vroom!'


class Car:
    def __init__(self, brand):
        self.brand = brand
        self.engine = Engine()           # A Car HAS an Engine

    def drive(self):
        return f'{self.brand}: {self.engine.start()}'


print(Car('Maruti').drive())

---

## Part 11: Practice Exercises

Try each exercise in the empty cell **before** looking at the solutions at the end.

---

### Exercise 1: Shapes ⭐
- Create a parent class `Shape` with a method `area()` that returns `0`
- Create `Rectangle(width, height)` and `Circle(radius)` that **override** `area()`
- Loop over a list of shapes and print each area (**polymorphism**!)

```python
shapes = [Rectangle(4, 5), Circle(3)]
# Rectangle area: 20
# Circle area: 28.27
```

In [ ]:
# Write your Exercise 1 solution here

### Exercise 2: Employees ⭐⭐
- `Employee(name, salary)` with a method `get_role()` returning `'Employee'`
- `Manager(name, salary, team_size)` — use `super().__init__()`, override `get_role()` to return `'Manager of 5 people'`
- `Developer(name, salary, language)` — override `get_role()` to return `'Python Developer'`

```python
Manager('Priya', 90000, 5).get_role()        # 'Manager of 5 people'
Developer('Amit', 70000, 'Python').get_role() # 'Python Developer'
```

In [ ]:
# Write your Exercise 2 solution here

### Exercise 3: Bank Accounts (multilevel) ⭐⭐⭐
- `BankAccount(holder, balance)` with `calculate_interest()` returning `0`
- `SavingsAccount(BankAccount)` — interest is **4%** of balance
- `PremiumSavings(SavingsAccount)` — interest is the Savings interest **plus a Rs.500 bonus** (use `super().calculate_interest()`!)

```python
SavingsAccount('Amit', 10000).calculate_interest()   # 400.0
PremiumSavings('Priya', 10000).calculate_interest()  # 900.0
```

In [ ]:
# Write your Exercise 3 solution here

---

### ✅ Solutions

*(Try the exercises first! Then run these to compare.)*

In [ ]:
# Solution 1: Shapes
class Shape:
    def area(self):
        return 0


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return round(3.14159 * self.radius ** 2, 2)


shapes = [Rectangle(4, 5), Circle(3)]
for shape in shapes:
    print(f'{type(shape).__name__} area: {shape.area()}')

In [ ]:
# Solution 2: Employees
class Employee:
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

    def get_role(self):
        return 'Employee'


class Manager(Employee):
    def __init__(self, name, salary, team_size):
        super().__init__(name, salary)
        self.team_size = team_size

    def get_role(self):
        return f'Manager of {self.team_size} people'


class Developer(Employee):
    def __init__(self, name, salary, language):
        super().__init__(name, salary)
        self.language = language

    def get_role(self):
        return f'{self.language} Developer'


staff = [Employee('Rahul', 40000), Manager('Priya', 90000, 5), Developer('Amit', 70000, 'Python')]
for person in staff:
    print(f'{person.name:6} -> {person.get_role()}')

In [ ]:
# Solution 3: Bank Accounts (multilevel)
class BankAccount:
    def __init__(self, holder, balance):
        self.holder = holder
        self.balance = balance

    def calculate_interest(self):
        return 0


class SavingsAccount(BankAccount):
    def calculate_interest(self):
        return self.balance * 0.04


class PremiumSavings(SavingsAccount):
    def calculate_interest(self):
        return super().calculate_interest() + 500     # parent's interest + bonus


accounts = [BankAccount('Rahul', 10000), SavingsAccount('Amit', 10000), PremiumSavings('Priya', 10000)]
for acc in accounts:
    print(f'{type(acc).__name__:15} {acc.holder:6} interest: Rs.{acc.calculate_interest()}')

---

## Part 12: Cheat Sheet — What We Learned Today

| Concept | Syntax | Meaning |
|---------|--------|---------|
| **Inheritance** | `class Child(Parent):` | Child gets everything the parent has |
| **super()** | `super().__init__(name, age)` | Call the parent's version of a method |
| **New method** | define a new name in the child | Only the child has it |
| **Override (replace)** | same method name in the child | Child's version runs instead |
| **Override (extend)** | `super().method()` inside the override | Parent's result + extra work |
| **Polymorphism** | `for a in animals: a.speak()` | Same call, different behaviour per object |
| **isinstance** | `isinstance(obj, Class)` | Is this object of that class (or a child)? |
| **issubclass** | `issubclass(Child, Parent)` | Is this class a child of that class? |
| **Multiple** | `class Duck(Flyer, Swimmer):` | Many parents, searched left → right (MRO) |

### The golden rules
1. Python finds methods by searching **child first, then parent** (then grandparent...).
2. If the child has its own `__init__`, call **`super().__init__(...)`**.
3. Only use inheritance when **"Child IS-A Parent"** makes sense. Otherwise use **HAS-A** (an attribute).
4. `_protected` → children may use it. `__private` → only the class itself.

---

### 🔮 Next: Day 20 — Abstraction
How to **force** every child class to write certain methods, using abstract classes.